### **CREATING AN INTERATOR FOR BACTH PROCESSING**

In [36]:
#!pip install PyPortfolioOpt
#!pip install scikit-learn
#!pip install stable-baselines3
#!pip install gymnasium
#!pip install pypfopt
#!pip install tqdm
#!pip install torch
#!pip install torchvision

import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from sklearn.preprocessing import StandardScaler
import torch
from datetime import datetime

class StockPortfolioIterator:
    def __init__(self, file_path: str, lookback_window: int = 30, min_history: int = 100, max_stocks: int = 100, train_years: int = 20):

        print(f"Loading data from {file_path}...")
        self.data = pd.read_csv(file_path)
        self.data['date'] = pd.to_datetime(self.data['date'])

        # Only use recent data (last few years)
        cutoff_date = self.data['date'].max() - pd.DateOffset(years=train_years)
        self.data = self.data[self.data['date'] >= cutoff_date]

        self.data = self.data.sort_values(['date', 'tic'])

        # Filter stocks with enough history
        stock_counts = self.data.groupby('tic').size()
        valid_stocks = stock_counts[stock_counts >= min_history].index

        # Select stocks with highest trading volume (more likely to be liquid)
        if len(valid_stocks) > max_stocks:
            avg_volumes = self.data[self.data['tic'].isin(valid_stocks)].groupby('tic')['volume'].mean()
            valid_stocks = avg_volumes.nlargest(max_stocks).index.tolist()

        self.data = self.data[self.data['tic'].isin(valid_stocks)]
        self.unique_dates = sorted(self.data['date'].unique())
        self.tickers = sorted(self.data['tic'].unique())
        self.num_stocks = len(self.tickers)
        self.lookback = lookback_window
        self.num_features = 7

        # Create ticker lookup for faster access
        self.ticker_indices = {ticker: i for i, ticker in enumerate(self.tickers)}

        self._add_technical_indicators()
        self._scale_features()

        # Store data in a more efficient format
        self.prepare_training_data()

        print(f"Loaded {self.num_stocks} stocks with {len(self.unique_dates)} trading days")
        print(f"Date range: {self.unique_dates[0]} to {self.unique_dates[-1]}")

    def _add_technical_indicators(self):

        print("Adding technical indicators...")
        indicators = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum']

        # Process one ticker at a time to avoid memory issues
        for tic in self.tickers:
            mask = self.data['tic'] == tic
            stock_data = self.data.loc[mask].copy()

            # Calculate technical indicators
            stock_data['Returns'] = stock_data['close'].pct_change()
            stock_data['Price_MA'] = stock_data['close'].rolling(window=20, min_periods=1).mean()
            stock_data['Momentum'] = stock_data['close'].pct_change(periods=10)
            stock_data['Volume_MA'] = stock_data['volume'].rolling(window=10, min_periods=1).mean()

            # RSI Calculation
            delta = stock_data['close'].diff()
            gain = (delta.clip(lower=0)).rolling(window=14, min_periods=1).mean()
            loss = (-delta.clip(upper=0)).rolling(window=14, min_periods=1).mean()
            rs = gain / (loss + 1e-8)
            stock_data['RSI'] = 100 - (100 / (1 + rs))

            # MACD Calculation
            exp1 = stock_data['close'].ewm(span=12, adjust=False).mean()
            exp2 = stock_data['close'].ewm(span=26, adjust=False).mean()
            stock_data['MACD'] = exp1 - exp2

            # Volatility
            stock_data['Volatility'] = stock_data['Returns'].rolling(window=30, min_periods=1).std()

            # Update the main dataframe
            self.data.loc[mask, indicators] = stock_data[indicators]

        # Fill missing values
        self.data = self.data.fillna(0)
        print("Technical indicators added successfully")

    def _scale_features(self):
        print("Scaling features...")
        feature_columns = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum']

        for tic in self.tickers:
            mask = self.data['tic'] == tic
            scaler = StandardScaler()
            self.data.loc[mask, feature_columns] = scaler.fit_transform(self.data.loc[mask, feature_columns])

    def prepare_training_data(self):
        print("Preparing training data...")
        # Pre-allocate arrays for features and prices
        self.all_features = {}
        self.all_prices = {}
        feature_columns = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum']

        # Process data by date for efficient access during training
        for date_idx, date in enumerate(self.unique_dates):
            date_mask = self.data['date'] == date
            date_data = self.data[date_mask]

            features = np.zeros((self.num_stocks, self.num_features))
            prices = np.zeros(self.num_stocks)

            for _, row in date_data.iterrows():
                ticker_idx = self.ticker_indices.get(row['tic'])
                if ticker_idx is not None:
                    features[ticker_idx] = row[feature_columns].values
                    prices[ticker_idx] = row['close']

            self.all_features[date] = features
            self.all_prices[date] = prices

    def __iter__(self):
        """Initialize iterator"""
        self.current_idx = self.lookback
        return self

    def __next__(self):
        """Get next batch of data"""
        if self.current_idx >= len(self.unique_dates):
            raise StopIteration

        current_date = self.unique_dates[self.current_idx]
        window_dates = self.unique_dates[self.current_idx - self.lookback:self.current_idx]

        # Get lookback window features
        features = np.zeros((self.num_stocks, self.lookback, self.num_features), dtype=np.float32)

        for i, date in enumerate(window_dates):
            if date in self.all_features:
                features[:, i, :] = self.all_features[date]

        prices = self.all_prices[current_date]

        self.current_idx += 1
        return {
            'features': features,
            'prices': prices,
            'date': current_date,
            'tickers': self.tickers
        }

    def reset(self):
        """Reset the iterator"""
        self.current_idx = self.lookback
        return self


### **CREATING AN ENVIRONMENT TO TRAIN PPO**

In [37]:
class PortfolioTradingEnv(gym.Env):
    def __init__(self, data_iterator, initial_cash=10000, transaction_cost=0.001):
        super().__init__()
        self.data_iterator = data_iterator
        self.initial_cash = initial_cash
        self.transaction_cost = transaction_cost
        self.num_stocks = data_iterator.num_stocks
        self.lookback = data_iterator.lookback
        self.num_features = data_iterator.num_features

        # Action space: [cash allocation, stock1 allocation, ..., stockN allocation]
        self.action_space = spaces.Box(
            low=0,
            high=1,
            shape=(self.num_stocks + 1,),
            dtype=np.float32
        )

        # Observation space: (num_stocks, lookback, num_features)
        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.num_stocks, self.lookback, self.num_features),
            dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):
        """Reset the environment"""
        # Reset the data iterator
        self.data_iterator.reset()
        self.cash = self.initial_cash
        self.holdings = np.zeros(self.num_stocks)

        try:
            self.current_step = next(self.data_iterator)
            return self._get_observation(), {}
        except StopIteration:
            # Handle the case where there's no more data
            return np.zeros((self.num_stocks, self.lookback, self.num_features), dtype=np.float32), {}

    def _get_observation(self):
        """Get the current observation"""
        return self.current_step['features'].astype(np.float32)

    def step(self, action):
        """Take a step in the environment"""
        # Normalize action
        action = np.clip(action, 0, 1)
        action_sum = action.sum()
        if action_sum > 0:
            action /= action_sum

        current_prices = self.current_step['prices']
        portfolio_value = self.cash + np.sum(self.holdings * current_prices)

        # Calculate target allocations
        target_values = action[1:] * portfolio_value
        current_values = self.holdings * current_prices
        delta_values = target_values - current_values

        # Apply transaction costs
        transaction_costs = np.abs(delta_values).sum() * self.transaction_cost

        # Update holdings and cash
        for i in range(self.num_stocks):
            if current_prices[i] > 0:  # Avoid division by zero
                self.holdings[i] += delta_values[i] / current_prices[i]

        self.cash = portfolio_value - np.sum(self.holdings * current_prices) - transaction_costs

        # Store current portfolio value for reward calculation
        old_portfolio_value = portfolio_value

        try:
            # Move to next time step
            self.current_step = next(self.data_iterator)
            new_prices = self.current_step['prices']
            done = False
        except StopIteration:
            new_prices = current_prices  # Use current prices if no more data
            done = True

        # Calculate reward (daily return)
        new_portfolio_value = self.cash + np.sum(self.holdings * new_prices)
        reward = (new_portfolio_value / old_portfolio_value) - 1

        # Apply penalty for excessive trading
        reward -= transaction_costs / old_portfolio_value

        return (
            self._get_observation() if not done else np.zeros_like(self.observation_space.sample()),
            float(reward),
            done,
            False,
            {"portfolio_value": new_portfolio_value}
        )



### **FUNCTION TO RECOMMEND STOCKS**

In [38]:
class PortfolioRecommender:
    def __init__(self, model_path, data_path, max_stocks=100, lookback=30):
        self.model = PPO.load(model_path)
        self.max_stocks = max_stocks
        self.lookback = lookback

        # Load the most recent data for prediction
        data = pd.read_csv(data_path)
        data['date'] = pd.to_datetime(data['date'])

        # Get the most recent date
        self.latest_date = data['date'].max()

        # Select top stocks by volume (same filtering as in training)
        recent_data = data[data['date'] >= (self.latest_date - pd.DateOffset(days=30))]
        avg_volumes = recent_data.groupby('tic')['volume'].mean()
        top_tickers = avg_volumes.nlargest(max_stocks).index.tolist()

        # Get the list of tickers (must match the order used in training)
        self.tickers = sorted(top_tickers)

        # Store the latest prices
        latest_data = data[data['date'] == self.latest_date]
        self.latest_prices = {row['tic']: row['close'] for _, row in latest_data.iterrows()
                              if row['tic'] in self.tickers}

        # Add technical indicators for better prediction (simplified version)
        self._prepare_features(data)

    def _prepare_features(self, data):
        # Filter for only the tickers we're using
        filtered_data = data[data['tic'].isin(self.tickers)]

        # Get the most recent dates for feature calculation
        recent_dates = sorted(filtered_data['date'].unique())[-self.lookback:]
        recent_data = filtered_data[filtered_data['date'].isin(recent_dates)]

        # Calculate basic features (simplified version of training features)
        features = np.zeros((self.max_stocks, self.lookback, 7), dtype=np.float32)

        for i, ticker in enumerate(self.tickers):
            if i >= self.max_stocks:
                break

            ticker_data = recent_data[recent_data['tic'] == ticker].sort_values('date')
            if len(ticker_data) == self.lookback:
                # Simple features (would need to match training features for best results)
                ticker_features = np.zeros((self.lookback, 7))
                ticker_features[:, 0] = ticker_data['close'].pct_change().fillna(0).values  # Returns
                ticker_features[:, 1] = ticker_data['volume'].values / ticker_data['volume'].mean()  # Volume normalized
                ticker_features[:, 2] = ticker_data['close'].values / ticker_data['close'].mean()  # Price normalized
                ticker_features[:, 3] = 50  # Placeholder for RSI
                ticker_features[:, 4] = 0   # Placeholder for MACD
                ticker_features[:, 5] = ticker_data['close'].pct_change().rolling(5).std().fillna(0).values  # Volatility
                ticker_features[:, 6] = ticker_data['close'].pct_change(5).fillna(0).values  # Momentum

                features[i] = ticker_features

        self.recent_features = features

    def recommend_portfolio(self, amount_cad):
        # Use the precomputed features for prediction
        action, _ = self.model.predict(self.recent_features, deterministic=True)

        # Normalize allocations
        action = np.clip(action, 0, 1)
        action_sum = action.sum()
        if action_sum > 0:
            action /= action_sum

        # Allocate cash based on model recommendation
        allocations = {}
        cash_allocation = action[0] * amount_cad

        # Process only up to the number of tickers we have
        for i, ticker in enumerate(self.tickers):
            if i >= len(self.tickers) or i+1 >= len(action):
                continue

            allocation = action[i+1] * amount_cad
            if allocation > 0 and ticker in self.latest_prices:
                price = self.latest_prices[ticker]
                shares = allocation / price if price > 0 else 0

                allocations[ticker] = {
                    "allocation_percent": float(action[i+1] * 100),
                    "allocation_cad": float(allocation),
                    "price_per_share": float(price),
                    "shares": float(shares)
                }

        return {
            "cash_percent": float(action[0] * 100),
            "cash_amount": float(cash_allocation),
            "stock_allocations": allocations,
            "total_amount": float(amount_cad),
            "recommendation_date": str(self.latest_date)
        }

### **TRAINING THE PPO MODEL**

In [39]:
def train_model(data_path, model_save_path, max_stocks=100, lookback=30, timesteps=200000):
    print(f"Starting training at {datetime.now()}")

    # Initialize components with more manageable parameters
    train_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=20
    )

    def make_env():
        return PortfolioTradingEnv(train_iter)

    train_env = DummyVecEnv([make_env])

    # Configure PPO model with more appropriate hyperparameters
    model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=3e-4,
        n_steps=1024,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        ent_coef=0.01,  # Encourage exploration
        verbose=1,
        device="auto"  # Use GPU if available
    )

    # Start training with progress monitoring
    print(f"Training model with {train_iter.num_stocks} stocks, each with {lookback} days of history")
    model.learn(total_timesteps=timesteps, progress_bar=True)

    # Save the trained model
    model.save(model_save_path)
    print(f"Training completed at {datetime.now()} and model saved to {model_save_path}!")

    return model

### **MAIN FUNCTION**

In [40]:
if __name__ == "__main__":
    # Set parameters
    DATA_PATH = "/content/historical_data.csv"
    MODEL_SAVE_PATH = "portfolio_ppo_model"
    MAX_STOCKS = 100  # Limit number of stocks to make training feasible
    LOOKBACK = 30     # Shorter lookback period
    TIMESTEPS = 200000  # Reasonable number of training steps

    # Train the model
    model = train_model(DATA_PATH, MODEL_SAVE_PATH, MAX_STOCKS, LOOKBACK, TIMESTEPS)

    # Example of how to use the recommender
    print("\nTesting portfolio recommendation with $10,000 CAD")
    recommender = PortfolioRecommender(MODEL_SAVE_PATH, DATA_PATH, max_stocks=MAX_STOCKS, lookback=LOOKBACK)
    recommendation = recommender.recommend_portfolio(10000)

    print(f"Recommended portfolio allocation (as of {recommendation['recommendation_date']}):")
    print(f"Cash: ${recommendation['cash_amount']:.2f} ({recommendation['cash_percent']:.2f}%)")
    print("\nStock allocations:")

    # Sort allocations by percentage (largest first)
    sorted_allocations = sorted(
        recommendation['stock_allocations'].items(),
        key=lambda x: x[1]['allocation_percent'],
        reverse=True
    )

    for ticker, details in sorted_allocations:
        if details['allocation_percent'] > 1.0:  # Only show significant allocations
            print(f"{ticker}: ${details['allocation_cad']:.2f} "
                  f"({details['allocation_percent']:.2f}%) - "
                  f"{details['shares']:.4f} shares @ ${details['price_per_share']:.2f}")

Starting training at 2025-03-01 00:41:06.441492
Loading data from /content/historical_data.csv...
Adding technical indicators...
Technical indicators added successfully
Scaling features...
Preparing training data...


Output()

/usr/local/lib/python3.11/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

Loaded 100 stocks with 5021 trading days
Date range: 2004-12-30 00:00:00 to 2024-12-30 00:00:00
Using cpu device
Training model with 100 stocks, each with 30 days of history
-----------------------------
| time/              |      |
|    fps             | 596  |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 1024 |
-----------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 192        |
|    iterations           | 2          |
|    time_elapsed         | 10         |
|    total_timesteps      | 2048       |
| train/                  |            |
|    approx_kl            | 0.13311423 |
|    clip_fraction        | 0.619      |
|    clip_range           | 0.2        |
|    entropy_loss         | -144       |
|    explained_variance   | -1.68      |
|    learning_rate        | 0.0003     |
|    loss                 | -1.58      |
|    n_updates            | 10         |
|

Training completed at 2025-03-01 01:09:43.981802 and model saved to portfolio_ppo_model!

Testing portfolio recommendation with $10,000 CAD
Recommended portfolio allocation (as of 2024-12-30 00:00:00):
Cash: $0.00 (0.00%)

Stock allocations:
BCE.TO: $398.95 (3.99%) - 12.3247 shares @ $32.37
BTCC.TO: $398.95 (3.99%) - 22.8233 shares @ $17.48
H.TO: $398.95 (3.99%) - 9.0057 shares @ $44.30
HND.TO: $398.95 (3.99%) - 38.7331 shares @ $10.30
IMO.TO: $398.95 (3.99%) - 4.5439 shares @ $87.80
T.TO: $398.95 (3.99%) - 20.6603 shares @ $19.31
TWM.TO: $398.95 (3.99%) - 3068.8550 shares @ $0.13
VET.TO: $398.95 (3.99%) - 30.9745 shares @ $12.88
XEG.TO: $398.95 (3.99%) - 23.7048 shares @ $16.83
CPX.TO: $390.20 (3.90%) - 6.1306 shares @ $63.65
WNDR.TO: $381.09 (3.81%) - 1314.1176 shares @ $0.29
LUN.TO: $371.76 (3.72%) - 30.4473 shares @ $12.21
BN.TO: $347.83 (3.48%) - 4.2233 shares @ $82.36
S.TO: $340.46 (3.40%) - 2127.8509 shares @ $0.16
GEI.TO: $335.32 (3.35%) - 13.7259 shares @ $24.43
DLR.TO: $297.1

PROMPTS USED : (FIRST PROMPT)

HI ChatGPT,


I am developing a project regarding Portfolio allocation using FinRL. In that I am specifically using  PPO for stock allocation.


Problem Statement : The user (daily traders) will be prompted with a input box that accepts amount in CAD, that they want to invest in Toronto stocks exchange. After clicking "Submit" button, the UI will give the user with recommended stocks to invest in a form of a portfolio using PPO from FinRL in the backend by analyzing the given stock dataset.


Dataset: The dataset that I have is scrapped from Yahoo finance. It has 1650 unique stocks from Toronto stock exchange. The date ranges from 2004-2024. It has columns like date, tic, open, close, low, high, volume. It has approximately 4 Million rows. The data is arranged by 'date' column and grouped by 'tic' column.


Task: I will give you the code that I have written so far. I want you to tell me if the environment, reward system and the action space is well enough to handle my large dataset. Also, If I run the the model training part I get the dimension mismatch error. Can you please make sure if the dimension return by iterator and dimension accepted by model are same. I also want you to modify the code and give me entire modified code so that this issue does not occur again.


The aim is to train PPO in the given dataset and recommend a portfolio of stocks based on the given input amount in CAD. The following code will contain stockdataiterator for batch processing, environment creation for PPO and training PPO.

LAST PROMPT:


Hi Chatgpt,
I am getting the following error when I run the model. Can you please tell me what I should do with the iterator function?

Process ForkServerProcess-1:
Traceback (most recent call last):
  File "/Users/rajkaranyp/anaconda3/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/rajkaranyp/anaconda3/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/rajkaranyp/Documents/PPO/myenv/lib/python3.11/site-packages/stable_baselines3/common/vec_env/subproc_vec_env.py", line 29, in _worker
    env = _patch_env(env_fn_wrapper.var())
                     ^^^^^^^^^^^^^^^^^^^^

TypeError: 'StockPortfolioIterator' object is not iterable